# P6-2 · Bài 8b · Preference tuning

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q datasets trl

## Lý thuyết

In [ ]:
{"prompt":   [{"role": "user", "content": "Đơn DH2051 giao chậm 3 ngày rồi, shop xử lý sao?"}],
 "chosen":   [{"role": "assistant", "content": "Xin lỗi anh/chị vì đơn DH2051 bị chậm. Em đã kiểm tra: ..."}],
 "rejected": [{"role": "assistant", "content": "Cảm ơn quý khách đã liên hệ. Chúng tôi luôn trân trọng ..."}]}

{'prompt': [{'role': 'user', 'content': 'Đơn DH2051 giao chậm 3 ngày rồi, shop xử lý sao?'}], 'chosen': [{'role': 'assistant', 'content': 'Xin lỗi anh/chị vì đơn DH2051 bị chậm. Em đã kiểm tra: ...'}], 'rejected': [{'role': 'assistant', 'content': 'Cảm ơn quý khách đã liên hệ. Chúng tôi luôn trân trọng ...'}]}


In [ ]:
# Cần GPU, model, LoRA và dataset ở các bước trước: chạy trên Colab
from trl import DPOTrainer, DPOConfig
args = DPOConfig(output_dir="qwen-dpo", beta=0.1, learning_rate=2e-5, num_train_epochs=1)
trainer = DPOTrainer(model=model, ref_model=None, args=args, train_dataset=ds["train"],
                     processing_class=tok, peft_config=lora_cfg)   # ref_model=None + LoRA: tham chiếu = model nền

## Bài tập 5: Chuẩn bị dataset cho DPO {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json
from datasets import Dataset

# file mẫu 3 dòng để chạy thử: một dòng hòa sẽ bị bỏ
demo = [{"prompt": "Đơn giao chậm, shop xử lý sao?", "answer_a": "Xin lỗi anh/chị, em kiểm tra ngay: ...", "answer_b": "Cảm ơn quý khách đã liên hệ ...", "preferred": "a"},
        {"prompt": "Bàn có kèm gương không?", "answer_a": "Có ạ.", "answer_b": "Có, gương 60×40 cm gắn sẵn trên bàn.", "preferred": "b"},
        {"prompt": "Shop mở cửa mấy giờ?", "answer_a": "8h–21h.", "answer_b": "Từ 8 giờ tới 21 giờ.", "preferred": "tie"}]
with open("prefs.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(d, ensure_ascii=False) + "\n" for d in demo)

rows = []
for line in open("prefs.jsonl", encoding="utf-8"):
    r = json.loads(line)
    if r["preferred"] not in ("a", "b"):
        continue
    good, bad = (r["answer_a"], r["answer_b"]) if r["preferred"] == "a" else (r["answer_b"], r["answer_a"])
    rows.append({
        "prompt": [{"role": "user", "content": r["prompt"]}],
        "chosen": [{"role": "assistant", "content": good}],
        "rejected": [{"role": "assistant", "content": bad}],
    })
ds = Dataset.from_list(rows).train_test_split(test_size=0.5, seed=0)    # dữ liệu thật: test_size=0.1
print(ds)
print(ds["train"][0]["chosen"])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 1
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 1
    })
})
[{'role': 'assistant', 'content': 'Có, gương 60×40 cm gắn sẵn trên bàn.'}]
